## **Algoritmo 1: CNN-1D**

In [7]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 1. CARGA Y PREPARACIÓN DE SECUENCIAS
file_path = '../Dataset/day_approach_maskedID_timeseries.csv'
df = pd.read_csv(file_path)

def create_sequences(df, window_size=21):
    X_seq = []
    y_labels = []
    metrics = ['total km', 'km sprinting', 'perceived exertion', 'perceived trainingSuccess']
    
    for athlete_id, group in df.groupby('Athlete ID'):
        group = group.reset_index(drop=True)
        injury_indices = group[group['injury'] == 1].index
        
        # Secuencias de LESIÓN
        for idx in injury_indices:
            if idx >= window_size:
                # Extraemos los 21 días brutos sin promediar
                sequence = group.loc[idx-window_size:idx-1, metrics].values
                X_seq.append(sequence)
                y_labels.append(1)
        
        # Secuencias de NO LESIÓN (Muestreo balanceado)
        no_injury_indices = group[group['injury'] == 0].index
        valid_indices = [i for i in no_injury_indices if i >= window_size]
        if valid_indices:
            # Tomamos 1.5 controles por cada lesionado para robustez
            sample_size = min(len(valid_indices), int(len(injury_indices) * 1.5))
            selected = np.random.choice(valid_indices, sample_size, replace=False)
            for idx in selected:
                sequence = group.loc[idx-window_size:idx-1, metrics].values
                X_seq.append(sequence)
                y_labels.append(0)

    return np.array(X_seq), np.array(y_labels)

X, y = create_sequences(df)

# 2. SPLIT Y ESCALADO (Vital para Redes Neuronales)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. DEFINICIÓN DE LA ARQUITECTURA CNN-1D
def build_cnn(input_shape):
    model = models.Sequential([
        # Capa 1: Detecta micro-patrones de 3 días
        layers.Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling1D(pool_size=2),
        
        # Capa 2: Detecta patrones semanales
        layers.Conv1D(filters=64, kernel_size=3, activation='relu'),
        layers.GlobalAveragePooling1D(), # Colapsa la serie temporal a un vector de rasgos
        
        # Capa Densa: Clasificación final
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid') # Probabilidad de lesión
    ])
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.Recall()])
    return model

# 4. ENTRENAMIENTO
input_shape = (X_train.shape[1], X_train.shape[2]) # (21, 4)
nn_model = build_cnn(input_shape)

history = nn_model.fit(
    X_train, y_train, 
    epochs=50, 
    batch_size=16, 
    validation_split=0.2,
    verbose=1
)

# 5. EVALUACIÓN
y_pred = (nn_model.predict(X_test) > 0.5).astype(int)
print("\nRESULTADOS RED NEURONAL (CNN-1D)")
print(classification_report(y_test, y_pred))

c:\Users\marcj\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.5646 - loss: 0.6703 - recall_1: 0.3736 - val_accuracy: 0.5498 - val_loss: 0.6982 - val_recall_1: 0.0000e+00
Epoch 2/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6482 - loss: 0.6285 - recall_1: 0.2995 - val_accuracy: 0.6364 - val_loss: 0.6430 - val_recall_1: 0.4020
Epoch 3/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6775 - loss: 0.6174 - recall_1: 0.4286 - val_accuracy: 0.5758 - val_loss: 0.6583 - val_recall_1: 0.0490
Epoch 4/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7003 - loss: 0.5860 - recall_1: 0.4560 - val_accuracy: 0.6190 - val_loss: 0.6332 - val_recall_1: 0.1863
Epoch 5/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.7188 - loss: 0.5650 - recall_1: 0.5549 - val_accuracy: 0.6147 - val_loss: 0.6337 - val_recall_1: 0.1667
Epoch 6/50
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7210 - loss: 0.5634 - recall_1: 0.5110 - val_accuracy: 0.6580 - val_loss: 0.6092 - val_rec

### 1. El Gran Triunfo: El Recall de Lesión ($0.83$)
Fíjate bien: el Random Forest tenía un Recall de **0.74**, pero la CNN ha subido al **0.83**.
* **¿Qué significa esto?** La Red Neuronal es mucho mejor "detectando" las lesiones. De 100 lesiones reales, la CNN encuentra 83, mientras que el Random Forest solo encontraba 74.
* **La contrapartida:** Al ser más "sensible", tiene más falsas alarmas (su precisión bajó a 0.73). Pero en medicina deportiva, solemos preferir avisar de más a que un atleta se rompa porque el modelo fue demasiado conservador.

### 2. Diagnóstico: El fantasma del Overfitting
Si miras las últimas épocas del entrenamiento:
* **Accuracy en entrenamiento:** $98.2\%$ (Casi perfecto).
* **Accuracy en validación:** $86.1\%$.
* **Conclusión:** El modelo se está "aprendiendo de memoria" los datos de entrenamiento. Ha empezado a memorizar el ruido. Para un proyecto universitario, esto se soluciona mencionando que se podría usar **Early Stopping** (parar el entrenamiento cuando la pérdida de validación deja de bajar) o aumentar el **Dropout**.

### 3. Comparativa Final

| Métrica | Random Forest (Baseline) | CNN-1D (Deep Learning) |
| :--- | :--- | :--- |
| **Accuracy Total** | **0.86** (Más estable) | 0.81 (Más volátil) |
| **Recall (Detectar lesiones)** | 0.74 | **0.83** (¡Ganador!) |
| **Complejidad** | Baja (Fácil de explicar) | Alta (Caja negra) |
| **Interpretación** | Sabemos qué variables importan | Aprende patrones ocultos |

---

### Conclusión 

> "Mientras que el modelo de **Random Forest** ofrece una mayor estabilidad global (Accuracy 0.86), la **Red Neuronal Convolucional (CNN-1D)** demuestra una capacidad superior para identificar patrones pre-lesivos, alcanzando un **Recall del 83%**. 
>
> Desde una perspectiva de Ingeniería Biomédica, la CNN es preferible como herramienta de cribado (screening), ya que minimiza los Falsos Negativos (lesiones no detectadas). Sin embargo, el desfase entre la precisión de entrenamiento (98%) y validación (86%) indica que el modelo está cerca del límite de capacidad para el volumen de datos actual, sugiriendo que un dataset más extenso permitiría a la arquitectura de Deep Learning superar definitivamente a los métodos clásicos."

## **Algoritmo 2: datos formato "flat"**

Al tener los datos ya estructurados en formato "flat" (una sola fila con el historial de los últimos 7 días).

El dataset ya está preparado para que cada fila sea una "foto" de la semana previa.

Para exprimir estos datos al máximo con una Red Neuronal (CNN), lo ideal es transformar esa fila plana en una matriz temporal y añadir métricas de ingeniería deportiva calculadas sobre esos 7 días.

**1. Preparación de la "Matriz Temporal" ($7 \times 10$)**

Tenemos 10 métricas base repetidas por 7 días. Vamos a ordenarlas cronológicamente para que la CNN pueda "leer" la progresión del lunes al domingo (el día .6 es el más cercano a la lesión).

In [12]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# 1. CARGA DE DATOS Y BALANCEO
file_path = '../Dataset/day_approach_maskedID_timeseries.csv'
df = pd.read_csv(file_path)

# Separamos lesionados de no lesionados
injury_df = df[df['injury'] == 1]
no_injury_df = df[df['injury'] == 0].sample(n=len(injury_df), random_state=42)

# Unimos para tener un dataset 50/50 y mezclamos
balanced_df = pd.concat([injury_df, no_injury_df]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Dataset balanceado: {len(balanced_df)} muestras ({len(injury_df)} lesiones, {len(no_injury_df)} controles)")



# 2. SELECCIÓN DE MÉTRICAS Y ORDENACIÓN
base_metrics = [
    'nr. sessions', 'total km', 'km Z3-4', 'km Z5-T1-T2', 'km sprinting', 
    'strength training', 'hours alternative', 'perceived exertion', 
    'perceived trainingSuccess', 'perceived recovery'
]
# Orden cronológico: de hace 7 días ('') al día previo ('.6')
suffixes = ['', '.1', '.2', '.3', '.4', '.5', '.6']



# 3. PREPARACIÓN DE DATOS (X e Y)
def prepare_data(df):
    sequences = []
    meta_features = []
    
    # Escalamos los datos antes de agruparlos (las redes odian números grandes)
    scaler = StandardScaler()
    
    for _, row in df.iterrows():
        day_data = []
        for s in suffixes:
            cols = [m + s for m in base_metrics]
            day_data.append(row[cols].values)
        
        sequences.append(day_data)
        
        # Calculamos métricas extra para la rama "Densa" (Ingeniería Biomédica)
        total_km = np.sum([row['total km' + s] for s in suffixes])
        exertion_avg = np.mean([row['perceived exertion' + s] for s in suffixes])
        # Mini ACWR: Carga de los últimos 2 días vs media semanal
        acute = (row['total km.5'] + row['total km.6']) / 2
        chronic = total_km / 7
        acwr = acute / (chronic + 0.001)
        
        meta_features.append([total_km, exertion_avg, acwr])
        
    return np.array(sequences), np.array(meta_features), df['injury'].values

X_seq, X_meta, y = prepare_data(balanced_df)

# Normalizamos las secuencias y los metadatos
# (Redimensionamos para escalar y luego volvemos a su forma)
X_seq_scaled = X_seq.reshape(-1, 10)
X_seq_scaled = StandardScaler().fit_transform(X_seq_scaled).reshape(-1, 7, 10)
X_meta_scaled = StandardScaler().fit_transform(X_meta)



# 4. SPLIT
X_train_seq, X_test_seq, y_train, y_test = train_test_split(X_seq_scaled, y, test_size=0.2, random_state=42, stratify=y)
X_train_meta, X_test_meta, _, _ = train_test_split(X_meta_scaled, y, test_size=0.2, random_state=42, stratify=y)



# 5. MODELO CNN-1D MIXTO
def build_final_model():
    # Rama Secuencial (CNN)
    seq_in = Input(shape=(7, 10), name="Sequence_Input")
    x = layers.Conv1D(32, kernel_size=3, activation='relu', padding='same')(seq_in)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, kernel_size=3, activation='relu', padding='same')(x)
    x = layers.GlobalAveragePooling1D()(x)
    
    # Rama de Metadatos (Densa)
    meta_in = Input(shape=(3,), name="Meta_Input")
    y_meta = layers.Dense(16, activation='relu')(meta_in)
    
    # Unión
    combined = layers.concatenate([x, y_meta])
    combined = layers.Dense(32, activation='relu')(combined)
    combined = layers.Dropout(0.4)(combined)
    output = layers.Dense(1, activation='sigmoid')(combined)
    
    model = models.Model(inputs=[seq_in, meta_in], outputs=output)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.Recall()])
    return model

model = build_final_model()



# 6. ENTRENAMIENTO
print("\nIniciando entrenamiento balanceado...")
history = model.fit(
    [X_train_seq, X_train_meta], y_train, 
    epochs=60, 
    batch_size=16, 
    validation_split=0.2,
    verbose=1
)



# 7. EVALUACIÓN FINAL
y_pred_prob = model.predict([X_test_seq, X_test_meta])
y_pred = (y_pred_prob > 0.5).astype(int)



print("\n" + "="*30)
print("RESULTADOS FINALES (BALANCEADOS)")
print("="*30)
print(classification_report(y_test, y_pred))

Dataset balanceado: 1166 muestras (583 lesiones, 583 controles)

Iniciando entrenamiento balanceado...
Epoch 1/60
47/47 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.5517 - loss: 0.7012 - recall_6: 0.5840 - val_accuracy: 0.5989 - val_loss: 0.6620 - val_recall_6: 0.6264
Epoch 2/60
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6134 - loss: 0.6573 - recall_6: 0.6080 - val_accuracy: 0.6096 - val_loss: 0.6603 - val_recall_6: 0.7582
Epoch 3/60
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6188 - loss: 0.6438 - recall_6: 0.6400 - val_accuracy: 0.5722 - val_loss: 0.6539 - val_recall_6: 0.5714
Epoch 4/60
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6456 - loss: 0.6393 - recall_6: 0.6587 - val_accuracy: 0.5561 - val_loss: 0.6648 - val_recall_6: 0.5714
Epoch 5/60
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6631 - loss: 0.6288 - recall_6: 0.6640 - val_accuracy: 0.5882 - val_loss: 0.6532 - val_recall_6: 0.6154
Epoch 6/60
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accu

la Accuracy de entrenamiento subía rápido (llegó a 0.78), mientras que la de validación se quedaba estancada en 0.59-0.60.

Esto es un caso de libro de Overfitting (Sobreajuste): el modelo es demasiado complejo para tan pocos datos (1,166 muestras) y está memorizando el ruido en lugar de aprender medicina deportiva.

## **Algoritmo 3: Reconstrucción de 21 días y CNN-1D**


In [13]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# 1. CARGA Y PREPARACIÓN DE DATOS DIARIOS
file_path = '../Dataset/day_approach_maskedID_timeseries.csv'
df = pd.read_csv(file_path)

# Nos quedamos solo con las 10 métricas base (sin sufijos) para tratarlas como una serie temporal pura
base_metrics = [
    'nr. sessions', 'total km', 'km Z3-4', 'km Z5-T1-T2', 'km sprinting', 
    'strength training', 'hours alternative', 'perceived exertion', 
    'perceived trainingSuccess', 'perceived recovery'
]

# 2. FUNCIÓN PARA CREAR VENTANAS DE 21 DÍAS
def create_21day_sequences(df, window_size=21):
    sequences = []
    labels = []
    
    # Ordenamos por Atleta y Fecha para que la secuencia sea lógica
    df = df.sort_values(['Athlete ID', 'Date'])
    
    for athlete_id, group in df.groupby('Athlete ID'):
        group = group.reset_index(drop=True)
        # Buscamos los índices donde hay lesión
        injury_indices = group[group['injury'] == 1].index
        
        for idx in injury_indices:
            if idx >= window_size:
                # Tomamos los 21 días previos a la lesión
                seq = group.loc[idx-window_size+1 : idx, base_metrics].values
                sequences.append(seq)
                labels.append(1)
                
                # MUESTREO DE CONTROL (No-lesión)
                # Para balancear, buscamos un periodo sin lesión del mismo atleta
                # que esté lejos del día de la lesión
                potential_controls = group[(group.index < idx - window_size) | (group.index > idx + window_size)]
                if len(potential_controls) > window_size:
                    c_idx = potential_controls.sample(1).index[0]
                    if c_idx >= window_size:
                        c_seq = group.loc[c_idx-window_size+1 : c_idx, base_metrics].values
                        sequences.append(c_seq)
                        labels.append(0)

    return np.array(sequences), np.array(labels)

X, y = create_21day_sequences(df)

# 3. NORMALIZACIÓN (Fundamental para Redes Neuronales)
# Escalamos los datos: la red aprende mucho mejor si los km y las notas están en el mismo rango
X_shape = X.shape # (N, 21, 10)
X_flattened = X.reshape(-1, 10)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_flattened).reshape(X_shape)

# 4. SPLIT
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, stratify=y, random_state=42)

# 5. ARQUITECTURA DE LA CNN PARA 21 DÍAS
def build_21day_cnn(input_shape):
    model = models.Sequential([
        Input(shape=input_shape),
        # Filtros para detectar patrones de 3 y 5 días
        layers.Conv1D(32, kernel_size=5, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        
        layers.Conv1D(64, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.GlobalMaxPooling1D(), # Captura el "pico" de carga más importante de los 21 días
        
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001), 
                  loss='binary_crossentropy', 
                  metrics=['accuracy', tf.keras.metrics.Recall()])
    return model

cnn_model = build_21day_cnn((21, 10))
cnn_model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=1)

# 6. RESULTADOS
y_pred = (cnn_model.predict(X_test) > 0.5).astype(int)
print(classification_report(y_test, y_pred))

Epoch 1/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.4971 - loss: 1.0300 - recall_7: 0.4472 - val_accuracy: 0.4457 - val_loss: 0.6959 - val_recall_7: 0.2169
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5747 - loss: 0.7445 - recall_7: 0.6111 - val_accuracy: 0.4857 - val_loss: 0.6979 - val_recall_7: 0.2048
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5848 - loss: 0.6726 - recall_7: 0.6556 - val_accuracy: 0.4914 - val_loss: 0.6982 - val_recall_7: 0.3133
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6408 - loss: 0.6396 - recall_7: 0.6667 - val_accuracy: 0.4629 - val_loss: 0.7042 - val_recall_7: 0.3373
Epoch 5/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6652 - loss: 0.5991 - recall_7: 0.7667 - val_accuracy: 0.5200 - val_loss: 0.7050 - val_recall_7: 0.5663
Epoch 6/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6983 - loss: 0.5603 - recall_7: 0.7694 - val_accuracy: 0.5029 - val_loss: 0.7065 - val_recall_7

## **Algoritmo 4: LSTM**

A diferencia de la CNN, la LSTM está diseñada para "recordar". Para un atleta, la fatiga del día 1 se va acumulando y la red decide qué olvidar y qué guardar hasta llegar al día 21. Es la arquitectura reina para series temporales.

In [14]:
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

def build_lstm_model(input_shape):
    model = models.Sequential([
        # LSTM: 32 unidades es suficiente para no sobreajustar
        layers.Input(shape=input_shape),
        LSTM(32, return_sequences=False, kernel_regularizer='l2'), 
        BatchNormalization(),
        Dropout(0.5),
        
        Dense(16, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    
    # Optimizer con learning rate muy bajo para no "saltarse" el mínimo
    opt = tf.keras.optimizers.Adam(learning_rate=0.0001)
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.Recall()])
    return model

# Early Stopping para evitar que el modelo se aprenda el ruido
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

lstm_model = build_lstm_model((21, 10))

print("\nEntrenando LSTM con memoria de 21 días...")
history = lstm_model.fit(
    X_train, y_train, 
    epochs=100, 
    batch_size=32, 
    validation_data=(X_test, y_test),
    callbacks=[early_stop],
    verbose=1
)

# Evaluación
y_pred = (lstm_model.predict(X_test) > 0.5).astype(int)
print("\n--- RESULTADOS LSTM ---")
print(classification_report(y_test, y_pred))


Entrenando LSTM con memoria de 21 días...
Epoch 1/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.4753 - loss: 1.1732 - recall_8: 0.2844 - val_accuracy: 0.4587 - val_loss: 0.8829 - val_recall_8: 0.2613
Epoch 2/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5201 - loss: 1.1548 - recall_8: 0.3634 - val_accuracy: 0.4587 - val_loss: 0.8815 - val_recall_8: 0.2883
Epoch 3/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5017 - loss: 1.1854 - recall_8: 0.2822 - val_accuracy: 0.4725 - val_loss: 0.8799 - val_recall_8: 0.3153
Epoch 4/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4501 - loss: 1.2007 - recall_8: 0.2867 - val_accuracy: 0.4862 - val_loss: 0.8786 - val_recall_8: 0.3514
Epoch 5/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4925 - loss: 1.1822 - recall_8: 0.3251 - val_accuracy: 0.4954 - val_loss: 0.8773 - val_recall_8: 0.3874
Epoch 6/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5166 - loss: 1.1349 - recall_8: 0.3160 - val_

(Accuracy 0.54, Recall 0.53) confirman algo fundamental: la LSTM (y las redes neuronales en general) no son capaces de encontrar una estructura temporal compleja en este dataset específico. No es culpa de tu código, es la naturaleza de los datos.

**1. El veredicto técnico: ¿Por qué la LSTM no "ve" nada?**
* **Ruido vs. Señal**: Las lesiones deportivas son eventos ruidosos. Factores fuera del CSV (calidad de sueño, estrés, nutrición, genética) influyen muchísimo. Las redes neuronales intentan modelar funciones matemáticas complejas; si hay mucho ruido y pocos datos (solo ~1,100 filas balanceadas), la red se confunde y acaba "adivinando" al azar (el 50% de precisión que ves).

* **Insuficiencia de datos**: Una LSTM brilla cuando tienes miles de secuencias (ej. 100,000 registros). Con 1,100 secuencias de 21 días, la red no tiene suficientes ejemplos para aprender la "curva de fatiga" de un atleta.

* **La superioridad de los Árboles**: El Random Forest no intenta entender la "forma" de la serie temporal, sino que busca umbrales: "Si los km suben más de X y la recuperación baja de Y, hay riesgo". En medicina deportiva, esas reglas simples suelen ser mucho más robustas que las redes profundas.

"Se realizó un estudio comparativo entre aprendizaje basado en árboles (Random Forest) y aprendizaje profundo secuencial (LSTM). Mientras que el Random Forest alcanzó una precisión del 86%, la arquitectura LSTM no logró superar el 54%. Esto sugiere que la predicción de lesiones en este dataset depende más de umbrales críticos de carga que de patrones secuenciales de larga duración. Para que un modelo de Deep Learning sea viable, se requeriría un volumen de datos significativamente mayor para capturar la variabilidad individual de los atletas."